# LSTM for Sequence Classification

In [1]:
import os

# to debug and pinpoint device side assertions
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive/data/archive-2/

test.csv  testdata.manual.2009.06.14.csv  train.csv  training.1600000.processed.noemoticon.csv


In [5]:
#! pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.2 MB/s eta 0:00:00


In [6]:
import tiktoken

In [7]:
tokenizer = tiktoken.get_encoding("gpt2")

In [8]:
VAL_SPLIT_FROM_TRAIN = 0.1
TRAIN_SPLIT_FROM_TRAIN = 1-VAL_SPLIT_FROM_TRAIN

In [9]:
def collate_fn(batch):
    texts = []
    labels = []

    for text, label in batch:
        texts.append(text)
        labels.append(label)
    return torch.stack(texts), torch.stack(labels)

In [10]:
class SentimentDataset(Dataset):
    def __init__(self, split, tokenizer, max_length=None, data_dir='/content/drive/MyDrive/data/archive-2', pad_token_id=50256):
        if split=='validation':
            self.split = 'train'
        else:
            self.split = split

        df = pd.read_csv(os.path.join(data_dir, f'{self.split}.csv'), encoding='unicode_escape')[['text', 'sentiment']]
        df = df.dropna()
        if split=='train' or split=='validation':
            df = df.sample(frac=1, random_state=123).reset_index(drop=True)

            df_negative = df.loc[df['sentiment']=='negative']
            df_neutral = df.loc[df['sentiment']=='neutral']
            df_positive = df.loc[df['sentiment']=='positive']

            if split=='train':
                df_negative=df_negative[:int(len(df_negative)*TRAIN_SPLIT_FROM_TRAIN)]
                df_neutral=df_neutral[:int(len(df_neutral)*TRAIN_SPLIT_FROM_TRAIN)]
                df_positive=df_positive[:int(len(df_positive)*TRAIN_SPLIT_FROM_TRAIN)]
            else:
                df_negative=df_negative[int(len(df_negative)*TRAIN_SPLIT_FROM_TRAIN):]
                df_neutral=df_neutral[int(len(df_neutral)*TRAIN_SPLIT_FROM_TRAIN):]
                df_positive=df_positive[int(len(df_positive)*TRAIN_SPLIT_FROM_TRAIN):]

            df = pd.concat([df_negative, df_neutral, df_positive])

        df = df.sample(frac=1, random_state=123).reset_index(drop=True)
        print(set(df['sentiment']))
        df['sentiment'] = df['sentiment'].map({"negative": 0, "neutral": 1, "positive": 2})
        print(set(df['sentiment']))
        self.df = df
        self.encoded_texts = []
        bad_indices = []

        for i in range(len(self.df)):
            if self.df.iloc[i]['sentiment']==np.nan or self.df.iloc[i]['sentiment'] not in {0, 1, 2}:
                bad_indices.append(i)
                continue
            try:
                self.encoded_texts.append(tokenizer.encode(df.loc[i]['text']))
            except TypeError:
                bad_indices.append(i)
                continue

        print(f'Bad Indices: {len(bad_indices)} out of {len(self.df)}')
        self.df.drop(bad_indices)

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

        #B
        self.encoded_texts = [
        encoded_text[:self.max_length]
        for encoded_text in self.encoded_texts
        ]

        #C
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.df.iloc[index]["sentiment"]
        try:
            return (
                torch.tensor(encoded, dtype=torch.long),
                torch.tensor(label, dtype=torch.long)
        )
        except:
            print(f'label: {label}')
            raise

    def __len__(self):
        return len(self.df)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
        if encoded_length > max_length:
            max_length = encoded_length
        return max_length

In [11]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout=0.5, pad_idx=0):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim,
                           hidden_dim,
                           num_layers=n_layers,
                           bidirectional=True,
                           dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # * 2 for bidirectional
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))

        lstm_out, (hidden, cell) = self.lstm(embedded)

        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

        output = self.fc(self.dropout(hidden))

        return output

In [12]:
def train_model(model, train_loader, val_loader, criterion, optimizer, n_epochs, device):
    best_val_loss = float('inf')

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0
        train_acc = 0

        for batch_idx, (text, labels) in enumerate(train_loader):
            text, labels = text.to(device), labels.to(device)

            optimizer.zero_grad()
            predictions = model(text)

            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc += (predictions.argmax(1) == labels).float().mean().item()

            if batch_idx % 100 == 0:
                print(f'Epoch: {epoch+1}/{n_epochs} | Batch: {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

        #validation
        model.eval()
        val_loss = 0
        val_acc = 0

        with torch.no_grad():
            for text, labels in val_loader:
                text, labels = text.to(device), labels.to(device)
                predictions = model(text)

                loss = criterion(predictions, labels)
                val_loss += loss.item()
                val_acc += (predictions.argmax(1) == labels).float().mean().item()

        train_loss /= len(train_loader)
        train_acc /= len(train_loader)
        val_loss /= len(val_loader)
        val_acc /= len(val_loader)

        print(f'Epoch: {epoch+1}/{n_epochs}')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pt')

In [13]:
def main():
    VOCAB_SIZE = 60000
    EMBEDDING_DIM = 300
    HIDDEN_DIM = 256
    OUTPUT_DIM = 3
    N_LAYERS = 2
    DROPOUT = 0.5
    BATCH_SIZE = 64
    N_EPOCHS = 10
    LEARNING_RATE = 0.001

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_dataset = SentimentDataset('train', tokenizer)

    # print(max(max(encoded_text) for encoded_text in train_dataset.encoded_texts))

    val_dataset = SentimentDataset('validation', tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = LSTMClassifier(
        vocab_size=VOCAB_SIZE,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        output_dim=OUTPUT_DIM,
        n_layers=N_LAYERS,
        dropout=DROPOUT
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_model(model, train_loader, val_loader, criterion, optimizer, N_EPOCHS, device)

In [14]:
if __name__ == '__main__':
    main()

{'positive', 'negative', 'neutral'}
{0, 1, 2}
Bad Indices: 0 out of 24730
50256
{'positive', 'negative', 'neutral'}
{0, 1, 2}
Bad Indices: 0 out of 2750
Epoch: 1/10 | Batch: 0/387 | Loss: 1.1018
Epoch: 1/10 | Batch: 100/387 | Loss: 1.0022
Epoch: 1/10 | Batch: 200/387 | Loss: 1.0645
Epoch: 1/10 | Batch: 300/387 | Loss: 0.8650
Epoch: 1/10
Train Loss: 1.0057 | Train Acc: 0.4870
Val Loss: 0.9731 | Val Acc: 0.5247
Epoch: 2/10 | Batch: 0/387 | Loss: 0.8143
Epoch: 2/10 | Batch: 100/387 | Loss: 0.9458
Epoch: 2/10 | Batch: 200/387 | Loss: 0.8376
Epoch: 2/10 | Batch: 300/387 | Loss: 0.9197
Epoch: 2/10
Train Loss: 0.8704 | Train Acc: 0.5934
Val Loss: 0.9335 | Val Acc: 0.5720
Epoch: 3/10 | Batch: 0/387 | Loss: 0.8834
Epoch: 3/10 | Batch: 100/387 | Loss: 0.8461
Epoch: 3/10 | Batch: 200/387 | Loss: 0.7016
Epoch: 3/10 | Batch: 300/387 | Loss: 0.8627
Epoch: 3/10
Train Loss: 0.7842 | Train Acc: 0.6466
Val Loss: 0.9323 | Val Acc: 0.5855
Epoch: 4/10 | Batch: 0/387 | Loss: 0.7184
Epoch: 4/10 | Batch: 100/